# Template code for running MAST to compare clones
Requires conda environment with rpy2; outputs are provided in data directory

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np 
import pandas as pd
import seaborn as sns

import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
large_data_dir = gf_utils.large_data_dir



In [ ]:
import rpy2
import anndata2ri
import os
import scanpy as sc
import pandas as pd
import numpy as np

# anndata2ri interconverts AnnData and Single Cell Experiment objects
anndata2ri.activate()
%load_ext rpy2.ipython
#%reload_ext rpy2.ipython

import matplotlib.pyplot as plt
import scipy.stats as st

import seaborn as sns
import glob
from functools import reduce


/tmp/ipykernel_603855/707336968.py:9: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [ ]:
lib = '2'
BC = 'BC004'

adata_dir = large_data_dir + 'MPN_WTA/MPN_' + lib + '_' + BC + '_genotyped.h5ad'
adata = sc.read_h5ad(adata_dir)

adata.obs['clone'] = pd.read_csv('../output/2_BC004_clone_assignments.txt', sep='\t', index_col=0)
adata = adata[adata.obs['cell_type'].isin(['HSC'])].copy()
adata = adata[adata.obs['clone'].isin(['MPL_PTPN11','MPL_RUNX1'])].copy()
adata = adata[adata.obs['pheno_leiden'].isin([4,0,3])].copy()

In [ ]:
%%capture

clone = 'MPL_RUNX1'

output_filename = f"MPL_RUNX1_vs_MPL_PTPN11_DE_2_BC004"

adata.obs['in_type'] = (adata.obs['clone'] == clone).map({True: 'in', False: 'out'})

sce_v2 = sc.AnnData(X = adata.X, 
                obs = pd.DataFrame({'in_type': adata.obs['in_type']}, 
                                    index = adata.obs.index),
                var = pd.DataFrame(index = adata.var.index))

%R library(scuttle)
%R library(MAST)
%R library(data.table)

# -i implies we are supplying sce_v2 as an input to R
%R -i sce_v2 -i output_filename

# set up a single cell experiment object in R, using the data stored in 'X' in sce anndata object
%R counts(sce_v2) <- assay(sce_v2, "X"); 
print("Finished Setup")

### run MAST
# identify in cells:
%R id_cells_in <- which(colData(sce_v2)$in_type == 'in')

# identify out cells:
%R id_cells_out <- which(colData(sce_v2)$in_type == 'out')

# Check the length of the two sets
%R print(length(id_cells_in))
%R print(length(id_cells_out))

# Create two dataframes: one with in cells and one with out cells
%R df1 <- t(data.frame(counts(sce_v2)[, id_cells_out])) # transpose because in sce genes are rows
%R df2 <- t(data.frame(counts(sce_v2)[, id_cells_in])) # transpose because in sce genes are rows

# We will use the function provided:
%R source("../../6_figure_MPN_AML_phylogeny/code/run_MAST.r")

# Syntax: %R pairwise_de(dataframe1, dataframe2, 'output_filename', 'output_folder')
# The output will be automatically saved as a csv file in the output_folder with output_filename
%R pairwise_de(df2, df1, output_filename, '../data/')

In [5]:
print(adata.obs['in_type'].value_counts())

in_type
in     1146
out     779
Name: count, dtype: int64


In [ ]:
lib = '1'
BC = 'BC002'

adata_dir = large_data_dir + 'MPN_WTA/MPN_' + lib + '_' + BC + '_genotyped.h5ad'
adata = sc.read_h5ad(adata_dir)

adata = adata[adata.obs['pheno_leiden'].isin([0,6,7,9,4,1])].copy()
adata = adata[adata.obs['cell_type'].str.contains('Classical monocyte')].copy()

adata.obs['clone'] = 'JAK2'
adata.obs.loc[adata.obs['pheno_leiden'] == 1, 'clone'] = 'ASXL1'

gf_utils.assign_genotypes(adata)

In [ ]:
%%capture

clone = 'ASXL1'

output_filename = f"ASXL_vs_JAK2_DE_1_BC002"

adata.obs['in_type'] = (adata.obs['clone'] == clone).map({True: 'in', False: 'out'})

sce_v2 = sc.AnnData(X = adata.X, 
                obs = pd.DataFrame({'in_type': adata.obs['in_type']}, 
                                    index = adata.obs.index),
                var = pd.DataFrame(index = adata.var.index))

%R library(scuttle)
%R library(MAST)
%R library(data.table)

# -i implies we are supplying sce_v2 as an input to R
%R -i sce_v2 -i output_filename

# set up a single cell experiment object in R, using the data stored in 'X' in sce anndata object
%R counts(sce_v2) <- assay(sce_v2, "X"); 
print("Finished Setup")

### run MAST
# identify in cells:
%R id_cells_in <- which(colData(sce_v2)$in_type == 'in')

# identify out cells:
%R id_cells_out <- which(colData(sce_v2)$in_type == 'out')

# Check the length of the two sets
%R print(length(id_cells_in))
%R print(length(id_cells_out))

# Create two dataframes: one with in cells and one with out cells
%R df1 <- t(data.frame(counts(sce_v2)[, id_cells_out])) # transpose because in sce genes are rows
%R df2 <- t(data.frame(counts(sce_v2)[, id_cells_in])) # transpose because in sce genes are rows

# We will use the function provided:
%R source("../../6_figure_MPN_AML_phylogeny/code/run_MAST.r")

# Syntax: %R pairwise_de(dataframe1, dataframe2, 'output_filename', 'output_folder')
# The output will be automatically saved as a csv file in the output_folder with output_filename
%R pairwise_de(df2, df1, output_filename, '../data/')

In [5]:
print(adata.obs['in_type'].value_counts())


in_type
out    5210
in     3978
Name: count, dtype: int64
